In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor,StackingRegressor, VotingClassifier
from sklearn.svm import SVC, SVR
from xgboost import XGBClassifier, XGBRegressor
from sklearn.linear_model import Ridge
#import optuna
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score, ConfusionMatrixDisplay, mean_squared_error, mean_absolute_error,r2_score
import joblib

In [ ]:
df = pd.read_csv('Diabetes_Data.csv')

df.head()

In [ ]:
print(df.shape)

df.describe()

In [ ]:
df['diabetes'].value_counts()

In [ ]:
df.isna().sum()

In [ ]:
sns.countplot(x = 'diabetes',data =df)

In [ ]:
for features in df.columns:
  print(features,df[features].unique())

In [ ]:
cat_col = ['gender', 'smoking_history', 'hypertension', 'heart_disease']
num_col = ['age', 'bmi', 'HbA1c_level']

In [ ]:
for col in cat_col:
  sns.countplot(df, x = df[col], hue = 'diabetes')
  plt.show()

In [ ]:
sns.pairplot(df, hue="diabetes")

In [ ]:
plt.figure(figsize=(12, 4))
for i, col in enumerate(num_col):
    plt.subplot(1, 3, i+1)
    sns.histplot(df[col], kde=True)
    plt.title(f'{col}')
plt.show()

In [ ]:
sns.boxplot(data=df, x="gender", y="age", hue="diabetes")

In [ ]:
sns.boxplot(data=df, x="hypertension", y="age", hue="diabetes")

In [ ]:
correlation = df.corr()

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.title('Correlation of Features')
plt.show()

In [ ]:
scaler = StandardScaler()

In [ ]:
X = df.drop("diabetes", axis = 1)
y = df["diabetes"]

In [ ]:
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test,y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

In [ ]:
rf = RandomForestClassifier(max_depth=5, n_estimators=1000, min_samples_split=10, max_leaf_nodes=10,random_state=42)
rf.fit(X_train, y_train)
rf.predict(X_test)

In [ ]:
def model_evaluation(predictions):
  print(classification_report(y_test, predictions))
  cm = confusion_matrix(y_test, predictions)
  disp = ConfusionMatrixDisplay(confusion_matrix=cm)
  disp.plot()
  plt.show()

In [ ]:
y_pred = rf.predict(X_test)
model_evaluation(y_pred)

In [ ]:
svc = SVC()
svc.fit(X_train, y_train)

In [ ]:

y_pred = svc.predict(X_test)
model_evaluation(y_pred)

In [ ]:
xgb = XGBClassifier()
xgb.fit(X_train, y_train)

In [ ]:
y_pred = xgb.predict(X_test)
model_evaluation(y_pred)

#Ensemble with voting classifier

In [ ]:
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svc', SVC(probability=True, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=200, random_state=42))
]


In [ ]:
voting_model = VotingClassifier(estimators=base_models, voting = 'hard')

In [ ]:
voting_model.fit(X_train, y_train)

In [ ]:
y_pred = voting_model.predict(X_test)
accuracy_score(y_test, y_pred)

In [ ]:
joblib.dump(voting_model, 'diabetes_model.pkl')

#Blood Glucose Prediction

In [ ]:
X = df.drop("blood_glucose_level", axis = 1)
y = df["blood_glucose_level"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state = 24)

In [ ]:
models = [Ridge, RandomForestRegressor, XGBRegressor, SVR]

def model_train_evaluate(model):
  model = model()
  model.fit(X_train, y_train)
  prediction = model.predict(X_test)
  mse = mean_squared_error(y_test, prediction) #

  return mean_squared_error(y_test, prediction, squared=False), r2_score(y_test, prediction), np.sqrt(mse)

In [ ]:
for model in models:
  print(model, model_train_evaluate(model))

#Using Stacking Regressor

In [ ]:
base_models = [
    ('ridge', Ridge(random_state=42)),
    ('svr', SVR()),
    ('rfr', RandomForestRegressor(n_estimators=100, random_state=42))
]

meta_model = XGBRegressor(n_estimators=1000, random_state=42)

In [ ]:
stacking_reg = StackingRegressor(estimators=base_models,final_estimator=meta_model,cv=5)

stacking_reg.fit(X_train, y_train)

In [ ]:
pred = stacking_reg.predict(X_test)

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
rmse

In [ ]:
joblib.dump(stacking_reg, 'BloodGlucose_model.pkl')